# 1.1 Prompting

How you phrase instructions changes what an agent produces. This notebook walks through four patterns, using a playful "capital of the Moon" question:

| Pattern | What it does |
| --- | --- |
| **No system prompt** | Default model behaviour (often refuses or answers factually). |
| **System prompt** | Sets role and task ("you are a science fiction writer"). |
| **Few-shot** | Shows example Q&A so the model copies the style. |
| **Structured prompt** | Asks for named fields in free text. |
| **Structured output** | Same fields, but as a Pydantic object you can use in code. |

## Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Basic prompting

First invoke with **no** system prompt. Then add a `system_prompt` on `create_agent` so the same user question is answered in character.

`response["messages"][1]` is the first AI turn after the human message at index `0`.

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(model="gpt-5-nano")

# Same user question reused in later cells
question = HumanMessage(content="What's the capital of the moon?")

response = agent.invoke(
    {"messages": [question]}
)

# messages[0] = HumanMessage, messages[1] = AI reply
print(response['messages'][1].content)

In [ ]:
# Role + instruction: the model should invent a city, not lecture astronomy
system_prompt = "You are a science fiction writer, create a capital city at the users request."

scifi_agent = create_agent(
    model="gpt-5-nano",
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

## Few-shot examples

Few-shot prompting puts **example turns** in the system prompt. The model infers format (short invented names like `Marsialis` / `Venusovia`) without you describing the format in prose.

In [ ]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia

"""

scifi_agent = create_agent(
    model="gpt-5-nano",
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

## Structured prompts

Ask for a **fixed outline** in the prompt (Name, Location, Vibe, Economy). The reply is still a string — useful for humans, but you have to parse it if you need fields in code.

In [ ]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries

"""

scifi_agent = create_agent(
    model="gpt-5-nano",
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

## Structured output

`response_format=CapitalInfo` tells the agent to fill a **Pydantic** model. After `invoke`, use `response["structured_response"]` — typed fields, no regex on free text.

That is the difference: structured *prompt* = shape in the text; structured *output* = shape in your program.

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel

# Schema the model must populate; field names become JSON keys
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

agent = create_agent(
    model='gpt-5-nano',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
    response_format=CapitalInfo  # validated object, not just a string
)

question = HumanMessage(content="What is the capital of The Moon?")

response = agent.invoke(
    {"messages": [question]}
)

response["structured_response"]

In [ ]:
# Attribute access on the Pydantic instance
response["structured_response"].name

In [ ]:
capital_info = response["structured_response"]

capital_name = capital_info.name
capital_location = capital_info.location

print(f"{capital_name} is a city located at {capital_location}")